This corresponds to task e) in the problem set: 

In [46]:
#IMPORTS: 
from tensorflow.keras.datasets import mnist
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import jax.numpy as jnp
import numpy as np
import neural_network_functions as nnf
import matplotlib.pyplot as plt
import time

import importlib
importlib.reload(nnf)

print(hasattr(nnf, "classification_accuracy"))


True


In [37]:

#Huge MNIST dataset: 
(X_train, y_train), (X_test, y_test) = mnist.load_data()
y_train = np.eye(10, dtype=np.float32)[y_train]
y_test = np.eye(10, dtype=np.float32)[y_test]
print(X_train.shape) # (60000, 28, 28)
print(y_train.shape) # (60000,)
print(X_test.shape)
print(y_test.shape)

#Smaller digits dataset used for testing: 
digits = load_digits()
X = digits.data      # shape (1797, 64)
y = digits.target    # labels 0-9
X_digits_train, X_digits_test, y_digits_train, y_digits_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
print(X_digits_train.shape)
print(y_digits_train.shape)

(60000, 28, 28)
(60000, 10)
(10000, 28, 28)
(10000, 10)
(1437, 64)
(1437,)


Scaling and centering the data: 

In [38]:
#digits data
scaler = StandardScaler()
X_digits_train = scaler.fit_transform(X_digits_train)
X_digits_test = scaler.transform(X_digits_test)

#MNIST data:
X_train = X_train.reshape(-1, 784).astype(np.float32)
X_test = X_test.reshape(-1, 784).astype(np.float32)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Prediction accuracy of digits dataset before training

In [47]:
##Written by copilot: 
network_input_size = 64
layer_output_sizes = [32, 10]

activation_funcs = [nnf.sigmoid, nnf.sigmoid]

layers = nnf.create_layers(
    network_input_size,
    layer_output_sizes,
)

# Predictions from untrained network
outputs = nnf.feed_forward(
    X_digits_test,
    layers,
    activation_funcs,
)

# Predicted digit = index of largest output neuron
y_pred = np.argmax(outputs, axis=1)

accuracy = np.mean(y_pred == y_digits_test)

print(f"Accuracy before training: {accuracy:.4f}")

Accuracy before training: 0.1639


Fixing ADAM as optimizer and network size as [784,200,200,10], and looking at accuracy for different activation functions on MNIST data: 

NOTE: Will take 30 minutes to run!

In [48]:
network_input_size = 784
layer_output_sizes = [200, 200, 10]

epochs = 20
batch_size = 128
learning_rate = 0.001

activation_experiments = {
    "Sigmoid": (
        nnf.sigmoid,
        nnf.sigmoid_der,
    ),
    "ReLU": (
        nnf.ReLU,
        nnf.ReLU_der,
    ),
    "Leaky ReLU": (
        nnf.leaky_ReLU,
        nnf.leaky_ReLU_der,
    ),
}

results = []
trained_models = {}

# Create the same initial parameters for every experiment
np.random.seed(42)

initial_layers = nnf.create_layers(
    network_input_size,
    layer_output_sizes,
)

for activation_name, (hidden_activation, hidden_derivative) in activation_experiments.items():
    print(f"\nTraining with {activation_name}...")

    # Make independent copies with identical initial parameters
    layers = [
        (W.copy(), b.copy())
        for W, b in initial_layers
    ]

    # Two hidden activations and one output activation
    activation_funcs = [
        hidden_activation,
        hidden_activation,
        nnf.sigmoid,
    ]

    activation_ders = [
        hidden_derivative,
        hidden_derivative,
        nnf.sigmoid_der,
    ]

    accuracy_before = nnf.classification_accuracy(
        layers,
        X_test,
        y_test,
        activation_funcs,
    )

    start_time = time.perf_counter()

    layers = nnf.train_sgd_adam(
        Xtrain=X_train,
        Ytrain=y_train,
        layers=layers,
        activation_funcs=activation_funcs,
        activation_ders=activation_ders,
        epochs=epochs,
        batch_size=batch_size,
        gamma=learning_rate,
        beta1=0.9,
        beta2=0.999,
        use_decaying_step_length=False,
        cost_der=nnf.mse_der,
    )

    training_time = time.perf_counter() - start_time

    train_accuracy = nnf.classification_accuracy(
        layers,
        X_train,
        y_train,
        activation_funcs,
    )

    test_accuracy = nnf.classification_accuracy(
        layers,
        X_test,
        y_test,
        activation_funcs,
    )

    results.append({
        "Activation": activation_name,
        "Accuracy before training": accuracy_before,
        "Training accuracy": train_accuracy,
        "Test accuracy": test_accuracy,
        "Training time (seconds)": training_time,
    })

    trained_models[activation_name] = {
        "layers": layers,
        "activation_funcs": activation_funcs,
    }

    print(f"Before training:  {accuracy_before:.4f}")
    print(f"Training accuracy: {train_accuracy:.4f}")
    print(f"Test accuracy:     {test_accuracy:.4f}")
    print(f"Training time:     {training_time:.1f} seconds")


Training with Sigmoid...


KeyboardInterrupt: 

Equivalent test for digits data. Runs much faster 